In [1]:
import pandas as pd #para la lectura de datos
import redis

# Conexión a Redis local
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
# Decodifica automáticamente las respuestas de Redis para que no devuelva siempre bytes.

# Probar la conexión
r.ping()

True

### Modelo 1: Estado del turno

In [2]:
# Cargamos los datos
df_turnos = pd.read_csv("./datos/turno",header=None)

In [3]:
for key in r.scan_iter("estado_turno:*"): # primero borramos datos antigüos por las dudas
    r.delete(key)

In [4]:
for _, row in df_turnos.iterrows(): # por cada fila del df de turnos
    r.set(                          # creamos una clave valor (turno,estado)
        f"estado_turno:{row[0]}:estado",
        row[5]
    )

In [5]:
# Ahora supongamos que una persona quiere saber si un turno no ha sido cancelado:
id_turno = 4
print('El estado del turno', id_turno, 'es:', r.get(f"estado_turno:{4}:estado"))


El estado del turno 4 es: programado


In [6]:
# Una persona cancelo un turno:
r.set(f"estado_turno:{id_turno}:estado",'cancelado')
print('El estado del turno', id_turno, 'es:', r.get(f"estado_turno:{4}:estado"))

El estado del turno 4 es: cancelado


### Modelo 2: Estado de consultorio

In [7]:
for key in r.scan_iter("estado_consultorio:*"): # primero borramos datos antigüos por las dudas
    r.delete(key)

In [ ]:
cant_consultorios = 42
for i in range(cant_consultorios): # al principio del dia todos los consultorios estan libres:
    r.set(f"estado_consultorio:{i}","libre")

In [12]:
# supongamos que llega un paciente y es atendido en el consultorio 3
print("El estado del consultorio 3 antes de modificarlo es", r.get(f'estado_consultorio:{3}'))
r.set(f"estado_consultorio:{3}","ocupado") # esto es lo que tendría que ejecutar recepción cada vez que se ocupa un consultorio
print("El estado del consultorio 3 después de modificarlo es", r.get(f'estado_consultorio:{3}'))
# podría hacer lo mismo cada vez que limpian un consultorio

El estado del consultorio 3 antes de modificarlo es libre
El estado del consultorio 3 después de modificarlo es ocupado


In [13]:
# si quiero una lista de todos los estados de los consultorios:
for key in r.scan_iter("estado_consultorio:*"): # por cada elemento
    print(key, "->", r.get(key))

estado_consultorio:21 -> libre
estado_consultorio:19 -> libre
estado_consultorio:23 -> libre
estado_consultorio:25 -> libre
estado_consultorio:31 -> libre
estado_consultorio:9 -> libre
estado_consultorio:36 -> libre
estado_consultorio:27 -> libre
estado_consultorio:32 -> libre
estado_consultorio:3: -> ocupado
estado_consultorio:29 -> libre
estado_consultorio:41 -> libre
estado_consultorio:4 -> libre
estado_consultorio:11 -> libre
estado_consultorio:0 -> libre
estado_consultorio:16 -> libre
estado_consultorio:40 -> libre
estado_consultorio:13 -> libre
estado_consultorio:18 -> libre
estado_consultorio:3:estado -> ocupado
estado_consultorio:5 -> libre
estado_consultorio:10 -> libre
estado_consultorio:37 -> libre
estado_consultorio:26 -> libre
estado_consultorio:15 -> libre
estado_consultorio:17 -> libre
estado_consultorio:3 -> ocupado
estado_consultorio:39 -> libre
estado_consultorio:24 -> libre
estado_consultorio:1 -> libre
estado_consultorio:33 -> libre
estado_consultorio:20 -> libre
es

### Cola: Lista de pacientes de un consultorio.

In [14]:
# Cargamos los datos
df_consultorio = pd.read_csv("./datos/datos_consultorio")
# estos datos fueron tomados de una consulta en la que se joineo turno con paciente y persona.
# con condiciones: fecha_turno='2026-09-01', estado = 'programado'.
# Son los turnos del dia de la fecha, el consultorio, la hora y nombre y apellido del cliente
print(df_consultorio.columns) 
# al comienzo del dia creamos un df por cada consultorio

Index(['id_turno', 'nombre', 'apellido', 'hora_turno', 'numero_consultorio'], dtype='object')


In [15]:
for key in r.scan_iter("consultorio:*"): # primero borramos datos antigüos por las dudas
    r.delete(key)

In [16]:
for _, row in df_consultorio.iterrows(): # Para cada numero de consultorio se crea una lista con los pacientes del dia ordenados (porque los datos venian ordenados)
    r.rpush(f'consultorio:{row["numero_consultorio"]}',row["nombre"]+ ' ' +row["apellido"])

In [17]:
# Con r.llen vemos cuantos elementos hay en la lista
print("Cantidad de turnos pendientes:", r.llen("consultorio:6"))
# Con r.lrange vemos todos los elementos de la lista
print("Pacientes pendientes:",r.lrange("consultorio:6", 0, -1))

Cantidad de turnos pendientes: 4
Pacientes pendientes: ['Juan Gabriel Diaz', 'Lautaro Gimenez', 'Gael Flores', 'Sofía Castillo']


In [19]:
print("Pacientes pendientes antes de atenderlo:",r.lrange("consultorio:6", 0, -1))
# Supongamos que el médico va a atender un paciente:
print(f"El siguiente paciente es {r.lpop("consultorio:6")}." )
# Verificamos que el paciente haya sido eliminado de la lista:
print("Pacientes pendientes después de antenderlo:",r.lrange("consultorio:6", 0, -1))

Pacientes pendientes antes de atenderlo: ['Lautaro Gimenez', 'Gael Flores', 'Sofía Castillo']
El siguiente paciente es Lautaro Gimenez.
Pacientes pendientes después de antenderlo: ['Gael Flores', 'Sofía Castillo']


Luego al principio del dia se volveria a agregar los pacientes para cada consultorio de la misma forma 

### Modelo 3, TTL 1: Sesiones de usuario

In [ ]:
# Supongamos que aproximadamente 1/4 de los pacientes accede a la web al mismo tiempo
muestra_sesiones = df_paciente.sample(frac=0.25, random_state=42) #tomamos aleatoriamente 25% de los pacientes


In [40]:
for key in r.scan_iter("sesion:*"): # primero borramos datos antigüos por las dudas
    r.delete(key)

In [32]:
id_sesion = '123ABC' #esto esta para que la clave de la sesion sea más parecida a los códigos que suelen verse en las sesiones.
for i, row in muestra_sesiones.iterrows():
    r.hset(                           # armo un hash con el id de la sesion como clave y me guardo todos los datos.
        f"sesion:{id_sesion+str(i)}",
        mapping={
            "cuil" : row['cuil'],
            "nombre": row['nombre'],
            "apellido": row['apellido'],
            "genero": row['genero'],
            "grupo_sanguineo": row['grupo_sanguineo'],
            "fecha_nacimiento": row['fecha_nacimiento'],
            "telefono": row['telefono'],
            "mail": row['mail'],
            "nombre_obra": row['nombre_obra']
        }
    )
    r.expire(f"sesion:{id_sesion+str(i)}",1800) # 30 minutos

In [33]:
# Supongamos que queremos saber cuanto tiempo le queda a la sesion de un usuario:
id_sesion = '123ABC1'
r.ttl(f"sesion:{id_sesion}") # nos da cuanto tiempo le queda a la sesión


1742

In [34]:
# definimos una función que dado un id de sesion nos printee el tiempo restante o su inexistencia
def estado(nombre,id):
    if r.ttl(f"{nombre}:{id}") == -2:
        print(f"El token {id} no existe o ya expiró.")
    elif r.ttl(f"{nombre}:{id}") == -1:
        print(f"La token {id} es persistente.")
    else:
        print("El token", id,"sigue vigente por",r.ttl(f"{nombre}:{id}"),"segundos.")


In [35]:
estado("sesion",id_sesion)

El token 123ABC1 sigue vigente por 1691 segundos.


In [36]:
# Supongamos que un usuario no se mantuvo inactivo por lo que corresponde extender la sesion:
print("Antes de la actualización:")
estado("sesion",id_sesion)
r.expire(f"sesion:{id_sesion}",1800)
print("Después de la actualización:")
estado("sesion",id_sesion)


Antes de la actualización:
El token 123ABC1 sigue vigente por 1682 segundos.
Después de la actualización:
El token 123ABC1 sigue vigente por 1800 segundos.


### TTL 2: Sesión en la computadora

In [41]:
for key in r.scan_iter("sesion_computadora:*"): # primero borramos datos antigüos por las dudas
    r.delete(key)

In [42]:
# De cada sesion nos guardamos el cuil del usuario, la hora de login y la hora de la última actividad
from datetime import datetime

id_sesion = '123ABC1'
r.hset(f'sesion_computadora:{id_sesion}',mapping={
            "cuil" : 12345678911,
            "datetime_login": datetime.now().isoformat(),
            "datetime_last_act": datetime.now().isoformat()
        })
r.expire(f"sesion_computadora:{id_sesion}",900) # 15 minutos

True

In [43]:
# Cuando el médico realiza alguna acción:
print("Antes de la actualización:")
estado("sesion_computadora",id_sesion) # obtenemos el tiempo restante
print("El datetime es:", r.hget(f'sesion_computadora:{id_sesion}',"datetime_last_act")) # obtenemos el valor

r.hset(f'sesion_computadora:{id_sesion}',mapping={"datetime_last_act":datetime.now().isoformat()}) # cambiamos el horario de última actividad
r.expire(f"sesion_computadora:{id_sesion}",900) # actualizamos el ttl a 15 minutos de vuelte

print("\nDespués de la actualización:")
estado("sesion_computadora",id_sesion) # obtenemos el nuevo tiempo restante
print("El datetime es:", r.hget(f'sesion_computadora:{id_sesion}',"datetime_last_act")) # obtenemos el nuevo valor

Antes de la actualización:
El token 123ABC1 sigue vigente por 893 segundos.
El datetime es: 2026-06-25T15:59:22.763858

Después de la actualización:
El token 123ABC1 sigue vigente por 900 segundos.
El datetime es: 2026-06-25T15:59:29.511973


### TTL 3: Reserva temporal de un turno

In [54]:
for key in r.scan_iter("turno_reservado:*"): # primero borramos datos antigüos por las dudas
    r.delete(key)

In [44]:
def agregar_reserva(datetime_turno,medico,id_paciente):
    clave = f"turno_reservado:medico:{medico}:{datetime_turno}"
    agregado = r.set(clave,id_paciente, nx=True, ex=600) #nx True hace que solo si guarde la clave si no existe,
    # La operación es atómica, por lo que si dos personas intentan reservar exactamente el mismo turno al mismo tiempo, sólo una lo conseguirá.
    # ex=600 es que tiene 10 minutos antes de expirar
    if not agregado: #si no se agrego
        print("El turno no se pudo reservar porque ya estaba reservado")

In [56]:
# Un paciente quiere un turno el 2026-25-12 a las 23:30:00
datetime_turno = datetime(2026, 12, 25, 11, 30)
medico = "1234567893"
id_paciente = "1234567894"
agregar_reserva(datetime_turno,medico,id_paciente) # agregamos el turno por primera vez
estado("turno_reservado:medico",f"{medico}:{datetime_turno}")

agregar_reserva(datetime_turno,medico,id_paciente) # intento agregar el mismo turno otra vez



El turno no se pudo reservar porque ya estaba reservado
El token 1234567893:2026-12-25 11:30:00 sigue vigente por 596 segundos.
El turno no se pudo reservar porque ya estaba reservado
